In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold,cross_val_score,RandomizedSearchCV,train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,f1_score,balanced_accuracy_score

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
train=pd.read_csv(r"F:\playground-series-s6e6\train.csv")
test=pd.read_csv(r"F:\playground-series-s6e6\test.csv")

In [3]:
train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [4]:
test.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [5]:
print(train.shape)
print(test.shape)

(577347, 12)
(247435, 11)


In [6]:
TARGET="class"
ID_COL="id"

train_ids=train[ID_COL]
test_ids=test[ID_COL]

X=train.drop(columns=[TARGET, ID_COL])
y=train[TARGET]
X_test=test.drop(columns=[ID_COL])

In [7]:
target_encoder=LabelEncoder()
y_encoded=target_encoder.fit_transform(y)

In [8]:
def feature_set_1(df):

    df = df.copy()
    df["u_g"]=df["u"] - df["g"]
    df["g_r"]=df["g"] - df["r"]
    df["r_i"]=df["r"] - df["i"]
    df["i_z"]=df["i"] - df["z"]
    df["u_r"]=df["u"] - df["r"]
    df["u_i"]=df["u"] - df["i"]
    df["u_z"]=df["u"] - df["z"]
    df["g_i"]=df["g"] - df["i"]
    df["g_z"]=df["g"] - df["z"]
    df["r_z"]=df["r"] - df["z"]
    return df

In [9]:
X_fs1=feature_set_1(X)
X_test_fs1=feature_set_1(X_test)

In [11]:
X_fs1.to_csv("X_fs1.csv", index=False)
X_test_fs1.to_csv("X_test_fs1.csv", index=False)

In [10]:
cat_cols=["spectral_type","galaxy_population"]

ohe = ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)],
    remainder="passthrough"
)

X_ohe=ohe.fit_transform(X_fs1)
X_test_ohe=ohe.transform(X_test_fs1)

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
X_mi=pd.get_dummies(X_fs1,drop_first=True)

mi_scores = mutual_info_classif(
    X_mi,
    y_encoded,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X_mi.columns,
    "MI": mi_scores
})

mi_df.sort_values(
    "MI",
    ascending=False
).head(30)

,Feature,MI
7,redshift,0.514989
16,g_z,0.406562
15,g_i,0.394854
14,u_z,0.382821
13,u_i,0.376867
19,spectral_type_M,0.334533
9,g_r,0.321157
12,u_r,0.310171
17,r_z,0.289004
21,galaxy_population_Red_Sequence,0.277107


In [16]:
et = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

et.fit(
    X_mi,
    y
)

et_df = pd.DataFrame({
    "Feature": X_mi.columns,
    "Importance": et.feature_importances_
})

et_df.sort_values(
    "Importance",
    ascending=False
).head(30)

,Feature,Importance
7,redshift,0.179178
19,spectral_type_M,0.133824
21,galaxy_population_Red_Sequence,0.079422
16,g_z,0.052864
3,g,0.048900
14,u_z,0.046796
15,g_i,0.045422
13,u_i,0.045260
6,z,0.038017
4,r,0.037010


In [17]:
lgbm_imp_model = LGBMClassifier(
    random_state=42,
    verbose=-1
)

lgbm_imp_model.fit(
    X_ohe,
    y
)

imp_df = pd.DataFrame({
    "Feature": ohe.get_feature_names_out(),
    "Importance": lgbm_imp_model.feature_importances_
})

imp_df.sort_values(
    "Importance",
    ascending=False
).head(30)

,Feature,Importance
6,remainder__alpha,1629
13,remainder__redshift,1619
7,remainder__delta,1608
22,remainder__g_z,394
9,remainder__g,390
12,remainder__z,388
14,remainder__u_g,361
8,remainder__u,344
10,remainder__r,343
11,remainder__i,292


In [17]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_encoded)),
    eval_metric="mlogloss",
    n_estimators=700,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=1.0,
    colsample_bynode=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    gamma=0.0,
    min_child_weight=1,
    max_delta_step=0,
    grow_policy="depthwise",
    max_leaves=0,
    max_bin=256,
    scale_pos_weight=1,
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
scores = cross_val_score(xgb,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy")
print("Mean CV:",scores.mean())
print("Std CV :",scores.std())

Mean CV: 0.9553188984122618
Std CV : 0.0007116167964913069


In [19]:
lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=len(np.unique(y_encoded)),
    boosting_type="gbdt",
    n_estimators=900,
    learning_rate=0.01,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    min_child_weight=1e-3,
    max_bin=255,
    device="gpu",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
lgbm_scores=cross_val_score(lgbm,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy",n_jobs=-1)
print("Mean CV :", lgbm_scores.mean())
print("Std CV  :", lgbm_scores.std())

Mean CV : 0.9564290463446523
Std CV  : 0.0006534594816852018


In [ ]:
from pytorch_tabular import TabularModel
from pytorch_tabular.config import (
    DataConfig,
    TrainerConfig,
    OptimizerConfig
)
from pytorch_tabular.models.ft_transformer.config import (
    FTTransformerConfig
)

cat_cols=["spectral_type","galaxy_population"]
num_cols=[c for c in X_fs1.columns if c not in cat_cols]
sample_idx=np.random.choice(len(X_fs1),300000,replace=False)
X_sample=X_fs1.iloc[sample_idx].copy()
y_sample=y_encoded[sample_idx]

X_train,X_valid,y_train,y_valid=train_test_split(X_sample,y_sample,test_size=0.2,stratify=y_sample,random_state=42)
train_df=X_train.copy()
train_df["target"]=y_train
valid_df = X_valid.copy()
valid_df["target"] = y_valid

data_config=DataConfig(target=["target"],
    continuous_cols=num_cols,
    categorical_cols=cat_cols)

trainer_config=TrainerConfig(
    batch_size=4096,
    max_epochs=30,
    accelerator="gpu",
    devices=1,
    progress_bar="rich")

optimizer_config=OptimizerConfig(
    optimizer="AdamW")

model_config=FTTransformerConfig(
    task="classification",
    input_embed_dim=32,
    num_heads=8,
    num_attn_blocks=4,
    ff_hidden_multiplier=4,
    learning_rate=3e-4,
    metrics=["accuracy"])


model=TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config)

model.fit(train=train_df,validation=valid_df)

preds=model.predict(valid_df)

pred_classes=preds["target_prediction"].values

score=balanced_accuracy_score(y_valid,pred_classes)

print("\n" + "=" * 50)
print(f"FT-Transformer Balanced Accuracy : {score:.6f}")
print("=" * 50)

print("\nPrediction Distribution")
print(pd.Series(pred_classes).value_counts(normalize=True))

2026-06-09 16:37:28,706 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-06-09 16:37:28,722 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-06-09 16:37:28,778 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task
2026-06-09 16:37:29,061 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-06-09 16:37:29,226 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-09 16:37:29,250 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
f:\Programs\PYTHON_310\lib\site-packages\pytorch_lightn

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ FTTransformerBackbone │  180 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding2dLayer      │  1.5 K │ train │     0 │
│ 2 │ _head            │ LinearHead            │     99 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss      │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 182 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 182 K                                                                                                
Total estimated model params size (MB): 0.730                                                                      
Modules in train mode: 90                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

f:\Programs\PYTHON_310\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

f:\Programs\PYTHON_310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

f:\Programs\PYTHON_310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

2026-06-09 16:54:26,769 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-06-09 16:54:26,771 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model



FT-Transformer Balanced Accuracy : 0.937429

Prediction Distribution
0    0.653333
1    0.201100
2    0.145567
Name: proportion, dtype: float64


In [ ]:
from sklearn.preprocessing import StandardScaler

X_ft = X_fs1.copy()
scaler=StandardScaler()
X_ft[num_cols] = scaler.fit_transform(X_ft[num_cols])

X_train, X_valid, y_train, y_valid = train_test_split(X_ft,y_encoded,test_size=0.2,stratify=y_encoded,random_state=42)
train_df=X_train.copy()
train_df["target"]=y_train

valid_df=X_valid.copy()
valid_df["target"]=y_valid

print("=" * 60)
print("Train Shape :", train_df.shape)
print("Valid Shape :", valid_df.shape)
print("=" * 60)


data_config = DataConfig(target=["target"],
    continuous_cols=num_cols,
    categorical_cols=cat_cols)

trainer_config = TrainerConfig(
    batch_size=512,
    max_epochs=100,
    accelerator="gpu",
    devices=1,
    progress_bar="rich",
    early_stopping="valid_loss",
    early_stopping_mode="min",
    early_stopping_patience=15)

optimizer_config=OptimizerConfig(optimizer="AdamW")

model_config=FTTransformerConfig(
    task="classification",
    learning_rate=1e-4,
    input_embed_dim=128,
    num_heads=16,
    num_attn_blocks=8,
    ff_hidden_multiplier=8,
    metrics=["accuracy"])


model=TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config)

print("\nStarting Training...\n")

model.fit(train=train_df,validation=valid_df)

print("\nTraining Finished\n")

preds = model.predict(valid_df)
pred_classes = preds["target_prediction"].values


score = balanced_accuracy_score(y_valid,pred_classes)

print("\n" + "=" * 60)
print(f"FT-Transformer Balanced Accuracy : {score:.6f}")
print("=" * 60)

print("\nPrediction Distribution")
print(preds["target_prediction"].value_counts(normalize=True).sort_index())

print("\nActual Distribution")
print(pd.Series(y_valid).value_counts(normalize=True).sort_index())
print("\nDone.")

2026-06-09 16:59:53,985 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
Seed set to 42
2026-06-09 16:59:54,000 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-06-09 16:59:54,091 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for classification task


Train Shape : (461877, 21)
Valid Shape : (115470, 21)

Starting Training...



2026-06-09 16:59:54,549 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: FTTransformerModel
2026-06-09 16:59:54,745 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-06-09 16:59:54,766 - {pytorch_tabular.tabular_model:677} - INFO - Training Started
f:\Programs\PYTHON_310\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory F:\playground-series-s6e6\Experiments\Fset1\saved_models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ FTTransformerBackbone │ 11.5 M │ train │     0 │
│ 1 │ _embedding_layer │ Embedding2dLayer      │  6.0 K │ train │     0 │
│ 2 │ _head            │ LinearHead            │    387 │ train │     0 │
│ 3 │ loss             │ CrossEntropyLoss      │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.5 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.5 M                                                                                               
Total estimated model params size (MB): 46.184                                                                     
Modules in train mode: 162                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

f:\Programs\PYTHON_310\lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

f:\Programs\PYTHON_310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

f:\Programs\PYTHON_310\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.